# Stage 3 on IBM Quantum simulation
Condensed from the Atomiq hardware notebook (same safety model, same Qiskit pins), but **self-contained**: it selects the six customers itself (classical exact Stage 2) and uses their `submission.csv` probabilities.

| Part | What it does |
|---|---|
| Stage 2 (classical, exact) | top-10 → best 6 under budget/capacity (210 subsets) |
| Stage 3 circuit | one `RY(2·arcsin√p)` per customer, qubit k ↔ k-th customer by ascending `customer_id`, success ⇔ ≥3 ones, Qiskit little-endian (q0 right-most) |
| Simulation | exact references, ideal Aer, **noisy Aer built from the backend's calibration**, readout mitigation, Wilson / bootstrap CIs |
| Hardware | best / middle / poor qubit groups, readout calibration circuits, dynamical decoupling + Pauli twirling, QPU-budget gates |

**Safety:** *Run All* never submits a job (`ENABLE_HARDWARE=False`). A job needs: the switch, the confirmation phrase, a known QPU balance, the job label in `CONFIRMED_JOBS`, enough budget, and a typed `YES`. The API key is read from Colab secret `IBM_QUANTUM_API_KEY` and is never printed or saved. If you have no key, a local fake-backend snapshot is used (clearly labelled).

## 1. Setup

In [ ]:
%pip install -q "python-dateutil>=2.9.0.post0,<3" "qiskit==2.2.3" "qiskit-aer==0.17.2" "qiskit-ibm-runtime==0.43.1"

In [ ]:
import os, json, math, time, itertools, platform
from dataclasses import dataclass, field, asdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import qiskit, qiskit_aer, qiskit_ibm_runtime
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
print(platform.python_version(), qiskit.__version__, qiskit_aer.__version__, qiskit_ibm_runtime.__version__)

# ================= CONFIG (edit here only) =================
ENABLE_HARDWARE = False                    # disarmed by default
REQUIRED_CONFIRMATION_TEXT = "SUBMIT TO IBM HARDWARE"
HARDWARE_CONFIRMATION_TEXT = ""            # type the phrase above ONLY for a deliberate hardware run
REMAINING_QPU_SECONDS = None               # read from the IBM dashboard (unknown => simulation only)
CONFIRMED_JOBS = set()                     # e.g. {"job1_stage3_baseline"}
MAX_NEW_JOBS_PER_RUN = 2
QPU_RESERVE_FRACTION, QPU_MIN_RESERVE_SECONDS = 0.30, 120.0
BUDGET_UNCERTAINTY_FACTOR, MAX_PLANNED_SECONDS_PER_JOB = 2.0, 60.0
FIXED_JOB_OVERHEAD_S, PER_CIRCUIT_OVERHEAD_S = 3.0, 0.5
MAX_WAIT_MINUTES, POLL_SECONDS = 30, 10
SHOTS = 8192                               # per circuit (also used for the readout calibrations)
RANDOM_SEED = 2026
MIN_BACKEND_QUBITS = 20
PREFERRED_BACKEND_NAME = None              # e.g. "ibm_fez"; None = least busy
STAGE3_IDS_OVERRIDE = None                 # e.g. ["C0028","C0031","C0079","C0081","C0327","C0338"] to force a set
SUCCESS_THRESHOLD = 3
API_KEY_SECRET_NAME, INSTANCE_SECRET_NAME = "IBM_QUANTUM_API_KEY", "IBM_QUANTUM_INSTANCE"
OUT = Path("results/ibm_hardware_quick"); RAW, PROC, FIG = OUT/"raw", OUT/"processed", OUT/"figures"
for d in (RAW, PROC, FIG): d.mkdir(parents=True, exist_ok=True)
def save_json(obj, path):
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w") as h: json.dump(obj, h, indent=2, default=lambda o: o.tolist() if isinstance(o, np.ndarray) else str(o))
    os.replace(tmp, path)

## 2. Stage 2 (classical exact) and the six Stage 3 customers

In [ ]:
def find(name):
    for d in [".", "/content", "/mnt/user-data/uploads", "/content/drive/MyDrive", ".."]:
        if os.path.exists(os.path.join(d, name)): return os.path.join(d, name)
    raise FileNotFoundError(name)
sub, data, cons = pd.read_csv(find("submission.csv")), pd.read_excel(find("stage_2_data.xlsx")), pd.read_excel(find("stage_2_cons.xlsx"))
cm = dict(zip(cons["symbol"].astype(str), cons["value"]))
N2, K2, BUDGET, CAPACITY = int(cm.get("N", 10)), int(cm.get("K", 6)), float(cm.get("C", 25)), float(cm.get("H", 22))
top = sub.sort_values(["probability", "customer_id"], ascending=[False, True]).head(N2).merge(data, on="customer_id").reset_index(drop=True)
top["net_value"] = top.probability * top.expected_deposit_value - top.contact_cost     # assumption: p*V - cost
best = max((c for c in itertools.combinations(range(N2), K2)
            if top.contact_cost[list(c)].sum() <= BUDGET and top.capacity_units[list(c)].sum() <= CAPACITY),
           key=lambda c: top.net_value[list(c)].sum())
selected = top.loc[list(best)]
if STAGE3_IDS_OVERRIDE:
    selected = sub[sub.customer_id.isin(STAGE3_IDS_OVERRIDE)]
    missing = set(STAGE3_IDS_OVERRIDE) - set(selected.customer_id)
    assert not missing, f"IDs not in submission.csv (no Stage 1 probability): {sorted(missing)}"
selected = selected.sort_values("customer_id").reset_index(drop=True)        # qubit k = k-th ascending ID
ids, p = selected.customer_id.tolist(), selected.probability.to_numpy(float); NQ = len(p)
assert NQ == 6 and ((p > 0) & (p < 1)).all()
print(pd.DataFrame({"qubit": [f"q{k}" for k in range(NQ)], "customer_id": ids, "p": p, "theta": 2*np.arcsin(np.sqrt(p))}).to_string(index=False))

## 3. Exact references (Poisson-binomial: DP and 64-state enumeration)

In [ ]:
def pmf_dp(p):
    d = np.zeros(len(p)+1); d[0] = 1.0
    for pi in p: d[1:] = d[1:]*(1-pi) + d[:-1]*pi; d[0] *= (1-pi)
    return d
POP = np.array([bin(x).count("1") for x in range(2**NQ)])
exact_outcomes = np.array([np.prod([p[q] if (x >> q) & 1 else 1-p[q] for q in range(NQ)]) for x in range(2**NQ)])  # bit q = qubit q
EXACT_TAIL = pmf_dp(p)[SUCCESS_THRESHOLD:].sum()
assert abs(EXACT_TAIL - exact_outcomes[POP >= SUCCESS_THRESHOLD].sum()) < 1e-12
print(f"EXACT P(K>={SUCCESS_THRESHOLD}) = {EXACT_TAIL:.10f}")

## 4. Circuits, ideal simulation and validation checks

In [ ]:
def bernoulli_circuit(p, measure=True):
    qc = QuantumCircuit(len(p), name="stage3")
    for q, pi in enumerate(p): qc.ry(2.0*math.asin(math.sqrt(pi)), q)
    if measure: qc.measure_all()                      # register name: "meas"
    return qc
def calibration_circuits(n):
    z = QuantumCircuit(n); z.measure_all()
    o = QuantumCircuit(n); o.x(range(n)); o.measure_all()
    return z, o
circ_A = bernoulli_circuit(p); circ_cal0, circ_cal1 = calibration_circuits(NQ)

def counts_to_vec(counts, n=NQ):
    v = np.zeros(2**n)
    for b, c in counts.items(): v[int(b.replace(" ", ""), 2)] += c
    return v / v.sum()
def tail_from_vec(v): return float(v[POP >= SUCCESS_THRESHOLD].sum())
def tail_from_counts(counts): return tail_from_vec(counts_to_vec(counts))

VALIDATION = []
def check(desc, ok, detail=""):
    VALIDATION.append(dict(check=desc, passed=bool(ok), detail=detail)); print(("[PASS] " if ok else "[FAIL] ") + desc, detail)
    if not ok: raise AssertionError(desc)
sv = Statevector(bernoulli_circuit(p, measure=False)).probabilities()
check("statevector == exact product distribution (qubit i = bit i, little-endian)", np.allclose(sv, exact_outcomes, atol=1e-12))
ideal = AerSimulator()
ideal_counts = ideal.run(transpile(circ_A, ideal), shots=SHOTS, seed_simulator=RANDOM_SEED).result().get_counts()
ideal_tail = tail_from_counts(ideal_counts)
check("ideal Aer within 4 binomial sigma of exact", abs(ideal_tail-EXACT_TAIL) < 4*math.sqrt(EXACT_TAIL*(1-EXACT_TAIL)/SHOTS), f"{ideal_tail:.4f} vs {EXACT_TAIL:.4f}")

## 5. IBM authentication and backend (read-only)

In [ ]:
def read_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception:
        return os.environ.get(name) or None

service, token = None, read_secret(API_KEY_SECRET_NAME)
if token:
    opts = {"channel": "ibm_quantum_platform", "token": token}
    inst = read_secret(INSTANCE_SECRET_NAME)
    if inst: opts["instance"] = inst
    try: service = QiskitRuntimeService(**opts); print("Connected to IBM Quantum.")
    except Exception as e: print("IBM login failed:", type(e).__name__, str(e).replace(token, "***"))
else:
    print("No IBM API key found -> simulation only with a local fake backend.")

if service is not None:
    cands = service.backends(operational=True, simulator=False, min_num_qubits=MIN_BACKEND_QUBITS)
    backend = (next((b for b in cands if b.name == PREFERRED_BACKEND_NAME), None) if PREFERRED_BACKEND_NAME else None) \
              or min(cands, key=lambda b: b.status().pending_jobs)
    IS_REAL_BACKEND = True
else:
    from qiskit_ibm_runtime import fake_provider
    backend = next(getattr(fake_provider, n)() for n in ("FakeFez", "FakeTorino", "FakeSherbrooke", "FakeBrisbane") if hasattr(fake_provider, n))
    IS_REAL_BACKEND = False
print("Backend:", backend.name, "| real hardware:", IS_REAL_BACKEND, "| qubits:", backend.num_qubits)

## 6. Calibration data and qubit groups (best / middle / poor)

In [ ]:
target = backend.target
TWO_Q = next(g for g in ("cz", "ecr", "cx") if g in target.operation_names)
def err(gate, qs):
    pr = target[gate].get(tuple(qs)) if gate in target.operation_names else None
    return float(pr.error) if pr is not None and pr.error is not None else float("nan")
calib = pd.DataFrame({"qubit": range(target.num_qubits)})
calib["readout_error"] = [err("measure", (q,)) for q in calib.qubit]; calib["sx_error"] = [err("sx", (q,)) for q in calib.qubit]
usable = calib.dropna(); usable = usable[usable.readout_error < 0.5].copy(); usable["score"] = usable.readout_error + usable.sx_error
ranked = usable.sort_values("score").reset_index(drop=True)
mid0, poor0 = len(ranked)//2 - NQ//2, int(0.9*len(ranked))
LAYOUTS = {"best": ranked.qubit[:NQ].tolist(), "middle": ranked.qubit[mid0:mid0+NQ].tolist(), "poor": ranked.qubit[poor0:poor0+NQ].tolist()}
print(f"two-qubit gate: {TWO_Q} (unused by Stage 3)  | median readout error: {usable.readout_error.median():.4f}")
pd.DataFrame({k: dict(qubits=v, mean_readout_err=calib.set_index("qubit").readout_error[v].mean(),
                      mean_sx_err=calib.set_index("qubit").sx_error[v].mean()) for k, v in LAYOUTS.items()}).T

## 7. Transpilation (one ISA circuit per layout) and hardware-compatibility checks

In [ ]:
PREPARED = {}
for lname, qubits in LAYOUTS.items():
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1, initial_layout=qubits, seed_transpiler=RANDOM_SEED)
    for tag, circ in (("A", circ_A), ("cal0", circ_cal0), ("cal1", circ_cal1)):
        isa = pm.run(circ); PREPARED[f"{tag}_{lname}"] = isa
        for inst in isa.data:
            if inst.operation.name == "barrier": continue
            qs = tuple(isa.find_bit(q).index for q in inst.qubits)
            assert target.instruction_supported(inst.operation.name, qs), f"{inst.operation.name} {qs} unsupported"
rows = []
for lab, isa in PREPARED.items():
    ops = isa.count_ops(); rows.append(dict(circuit=lab, depth=isa.depth(), two_qubit_gates=sum(v for k, v in ops.items() if k in ("cz", "ecr", "cx")), ops=dict(ops)))
print(pd.DataFrame(rows).to_string(index=False)); print("All instructions native on their physical qubits; Stage 3 needs no two-qubit gates.")

## 8. Noisy Aer simulation from the backend calibration + readout mitigation

In [ ]:
@dataclass
class RunRecord:
    label: str; source: str; layout: str; shots: int; counts: dict
    backend: str = ""; job_id: str = ""; job_label: str = ""; suppression: str = "none"
RUNS = {}
noisy = AerSimulator.from_backend(backend)
for i, (lab, isa) in enumerate(PREPARED.items()):
    c = noisy.run(isa, shots=SHOTS, seed_simulator=RANDOM_SEED + i).result().get_counts()
    RUNS[f"noisy_sim:{lab}"] = RunRecord(lab, "noisy_sim", lab.split("_")[-1], SHOTS, dict(c), backend=f"Aer.from_backend({backend.name})")

# ---- readout mitigation: tensor-product confusion-matrix inversion from |0..0> and |1..1> calibrations ----
def apply_per_qubit(vec, mats):
    t = vec.reshape((2,)*NQ)
    for q in range(NQ):
        ax = NQ-1-q; t = np.moveaxis(np.tensordot(mats[q], t, axes=([1], [ax])), 0, ax)
    return t.reshape(-1)
def confusion(zero_vec, one_vec):
    idx = np.arange(2**NQ); mats = []
    for q in range(NQ):
        bit = ((idx >> q) & 1).astype(bool)
        f01, f10 = zero_vec[bit].sum(), one_vec[~bit].sum()
        mats.append(np.array([[1-f01, f10], [f01, 1-f10]]))
    return mats
def mitigate(measured, zero_vec, one_vec):
    out = np.clip(apply_per_qubit(measured, [np.linalg.inv(m) for m in confusion(zero_vec, one_vec)]), 0, None)
    return out / out.sum()
def mitigated_vec(run):
    z = RUNS.get(run.source + ":cal0_" + run.label.split("_", 1)[1]) if run.source != "hardware" else RUNS.get(f"hardware:cal0_{run.label.split('_',1)[1]}:{run.job_label}")
    o = RUNS.get(run.source + ":cal1_" + run.label.split("_", 1)[1]) if run.source != "hardware" else RUNS.get(f"hardware:cal1_{run.label.split('_',1)[1]}:{run.job_label}")
    if z is None or o is None or run.suppression != "none": return None
    return mitigate(counts_to_vec(run.counts), counts_to_vec(z.counts), counts_to_vec(o.counts))
print({k: round(tail_from_counts(r.counts), 4) for k, r in RUNS.items() if r.label.startswith("A_")}, "| exact", round(EXACT_TAIL, 4))

## 9. Hardware execution (explicitly gated)
Planned jobs (all Stage 3, no two-qubit gates): **job1** best qubits baseline · **job2** best qubits with dynamical decoupling + Pauli twirling · **job3** middle and poor groups. Every job also measures the |0…0⟩ and |1…1⟩ readout calibrations in the same job. The job ID is written to `raw/job_registry.json` **before** waiting, so a disconnect never loses (or resubmits) a job: just re-run the notebook.

In [ ]:
JOBS = {"job1_stage3_baseline":       dict(layouts=["best"], suppression=False),
        "job2_stage3_suppressed":     dict(layouts=["best"], suppression=True),
        "job3_stage3_other_layouts":  dict(layouts=["middle", "poor"], suppression=False)}
REGISTRY_PATH = RAW / "job_registry.json"
REGISTRY = json.load(open(REGISTRY_PATH)) if REGISTRY_PATH.exists() else {}
NEW_JOBS = 0

def pub_labels(job_label):
    return [f"{tag}_{l}" for l in JOBS[job_label]["layouts"] for tag in ("A", "cal0", "cal1")]
def planned_seconds(job_label):
    total = FIXED_JOB_OVERHEAD_S
    delay = getattr(backend, "default_rep_delay", None) or 250e-6
    for lab in pub_labels(job_label):
        total += PER_CIRCUIT_OVERHEAD_S + SHOTS * (delay + 150e-6)
    return min(total * BUDGET_UNCERTAINTY_FACTOR, 1e9)
def used_seconds():
    return sum(r.get("actual_s") or r.get("planned_s", 0) for r in REGISTRY.values())
def gate_failures(job_label):
    f = []
    if not ENABLE_HARDWARE: f.append("ENABLE_HARDWARE is False")
    if HARDWARE_CONFIRMATION_TEXT != REQUIRED_CONFIRMATION_TEXT: f.append("confirmation phrase not set")
    if not IS_REAL_BACKEND: f.append("no IBM connection (fake backend)")
    if REMAINING_QPU_SECONDS is None: f.append("REMAINING_QPU_SECONDS unknown")
    if job_label not in CONFIRMED_JOBS: f.append("job not in CONFIRMED_JOBS")
    if NEW_JOBS >= MAX_NEW_JOBS_PER_RUN: f.append("MAX_NEW_JOBS_PER_RUN reached")
    if REMAINING_QPU_SECONDS is not None:
        reserve = max(QPU_MIN_RESERVE_SECONDS, QPU_RESERVE_FRACTION * REMAINING_QPU_SECONDS)
        spendable = REMAINING_QPU_SECONDS - reserve - used_seconds()
        need = planned_seconds(job_label)
        if need > MAX_PLANNED_SECONDS_PER_JOB: f.append(f"planned {need:.0f}s exceeds per-job cap")
        if need > spendable: f.append(f"planned {need:.0f}s exceeds spendable {spendable:.0f}s")
    return f

def state_of(job): s = job.status(); return str(getattr(s, "name", s)).upper()
def collect_job(job_label):
    rec = REGISTRY[job_label]; job = service.job(rec["job_id"]); st = state_of(job)
    rec["state"] = st
    if st in ("DONE", "COMPLETED"):
        res = job.result(); cfg = JOBS[job_label]
        for lab, pub in zip(rec["pub_labels"], res):
            counts = pub.data.meas.get_counts()
            RUNS[f"hardware:{lab}:{job_label}"] = RunRecord(lab, "hardware", lab.split("_")[-1], sum(counts.values()), dict(counts),
                backend=rec["backend"], job_id=rec["job_id"], job_label=job_label, suppression="dd+twirl" if cfg["suppression"] else "none")
        try: rec["actual_s"] = float(job.usage())
        except Exception:
            try: rec["actual_s"] = float(job.metrics()["usage"]["quantum_seconds"])
            except Exception: rec["actual_s"] = rec.get("planned_s")
    save_json(REGISTRY, REGISTRY_PATH); return st

def run_job(job_label):
    global NEW_JOBS
    if job_label in REGISTRY: st = collect_job(job_label); print(job_label, "->", st); return st
    fails = gate_failures(job_label)
    if fails: print(f"[{job_label}] SIMULATION ONLY. Reasons: " + "; ".join(fails)); return "skipped"
    if input(f"Submit {job_label} to {backend.name} (~{planned_seconds(job_label):.0f}s planned)? Type YES: ").strip() != "YES":
        print("not confirmed"); return "declined"
    cfg = JOBS[job_label]; sampler = SamplerV2(mode=backend)
    if cfg["suppression"]:
        sampler.options.dynamical_decoupling.enable = True; sampler.options.dynamical_decoupling.sequence_type = "XpXm"
        sampler.options.twirling.enable_gates = True; sampler.options.twirling.enable_measure = True
        sampler.options.twirling.num_randomizations = "auto"; sampler.options.twirling.shots_per_randomization = "auto"
    labels = pub_labels(job_label)
    job = sampler.run([PREPARED[l] for l in labels], shots=SHOTS)
    REGISTRY[job_label] = dict(job_id=job.job_id(), backend=backend.name, pub_labels=labels, planned_s=planned_seconds(job_label),
                               layouts={l: LAYOUTS[l] for l in cfg["layouts"]}, state="SUBMITTED"); NEW_JOBS += 1
    save_json(REGISTRY, REGISTRY_PATH)                               # persisted BEFORE waiting
    t0 = time.time()
    while time.time() - t0 < MAX_WAIT_MINUTES*60 and state_of(job) not in ("DONE", "COMPLETED", "ERROR", "CANCELLED", "FAILED"): time.sleep(POLL_SECONDS)
    st = collect_job(job_label); print(job_label, "->", st, "| job id", job.job_id()); return st

print("Mode:", "HARDWARE POSSIBLE" if not gate_failures("job1_stage3_baseline") else "SIMULATION ONLY", "| registry jobs:", list(REGISTRY))

In [ ]:
# Collect anything already in the registry (resume after disconnect), then run the jobs you confirmed.
for jl in JOBS: run_job(jl)

## 10. Results table with 95 % confidence intervals

In [ ]:
def wilson(k, n):
    ci = stats.binomtest(int(k), int(n)).proportion_ci(confidence_level=0.95, method="wilson"); return ci.low, ci.high
def bootstrap_tail(run, reps=500, seed=RANDOM_SEED):
    g = np.random.default_rng(seed)
    z = RUNS.get((run.source + ":cal0_" if run.source != "hardware" else "hardware:cal0_") + run.label.split("_", 1)[1] + ("" if run.source != "hardware" else ":" + run.job_label))
    o = RUNS.get((run.source + ":cal1_" if run.source != "hardware" else "hardware:cal1_") + run.label.split("_", 1)[1] + ("" if run.source != "hardware" else ":" + run.job_label))
    vs = [counts_to_vec(x.counts) for x in (run, z, o)]; ns = [sum(x.counts.values()) for x in (run, z, o)]; out = []
    for _ in range(reps):
        r = [g.multinomial(n, v) / n for n, v in zip(ns, vs)]; out.append(tail_from_vec(mitigate(*r)))
    return np.percentile(out, [2.5, 97.5])

rows = [dict(name="Ideal Aer", source="ideal_sim", layout="-", mitigation="none", est=ideal_tail, lo=wilson(ideal_tail*SHOTS, SHOTS)[0], hi=wilson(ideal_tail*SHOTS, SHOTS)[1], job="")]
for key, r in RUNS.items():
    if not r.label.startswith("A_"): continue
    t = tail_from_counts(r.counts); lo, hi = wilson(round(t*r.shots), r.shots)
    rows.append(dict(name=f"{r.source} ({r.suppression})", source=r.source, layout=r.layout, mitigation="none", est=t, lo=lo, hi=hi, job=r.job_label))
    mv = mitigated_vec(r)
    if mv is not None:
        bl, bh = bootstrap_tail(r); rows.append(dict(name=f"{r.source} + readout mitigation", source=r.source, layout=r.layout, mitigation="readout", est=tail_from_vec(mv), lo=bl, hi=bh, job=r.job_label))
table = pd.DataFrame(rows)
table["abs_error"] = (table.est - EXACT_TAIL).abs()
table["exact_in_CI"] = (table.lo <= EXACT_TAIL) & (EXACT_TAIL <= table.hi)
print(f"Exact P(K>={SUCCESS_THRESHOLD}) = {EXACT_TAIL:.6f}"); table.round(4)

## 11. Figure and saved artifacts

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.5*len(table)+1.8))
col = {"ideal_sim": "#0072B2", "noisy_sim": "#E69F00", "hardware": "#D55E00"}
for i, r in enumerate(table.itertuples()):
    ax.errorbar(r.est, i, xerr=[[r.est-r.lo], [r.hi-r.est]], fmt="o", color=col.get(r.source, "gray"), capsize=3)
ax.axvline(EXACT_TAIL, color="k", ls="--", label="exact")
ax.set_yticks(range(len(table))); ax.set_yticklabels([f"{r.name} [{r.layout}]" for r in table.itertuples()], fontsize=8)
ax.set_xlabel(f"P(K>={SUCCESS_THRESHOLD})"); ax.legend(); ax.set_title("Stage 3: exact vs simulation vs hardware (95% CI, shot noise only)")
fig.savefig(FIG/"stage3_comparison.png", dpi=300, bbox_inches="tight"); plt.show()

for k, r in RUNS.items(): save_json(asdict(r), RAW / ("counts__" + k.replace(":", "__") + ".json"))
table.to_csv(PROC/"stage3_results.csv", index=False); pd.DataFrame(VALIDATION).to_csv(PROC/"validation_log.csv", index=False)
save_json(dict(customer_ids=ids, probabilities=p, exact_tail=EXACT_TAIL, qubit_order="ascending customer_id, q0 = right-most bit", layouts=LAYOUTS), PROC/"stage3_inputs.json")
n_hw = sum(r.source == "hardware" for r in RUNS.values())
print("Hardware records:", n_hw, "| artifacts in", OUT)
if n_hw == 0: print("No hardware result in this run: everything above is simulation. Do not report a hardware claim.")